# Batch Inference for Bank Marketing Prediction

## 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import json

## 2. Load Trained Model

In [ ]:
model = None
model_path = '../training_pipeline/best_tuned_model.joblib'
feature_names_path = '../feature_pipeline/selected_feature_names.json'
trained_model_features = []

if os.path.exists(model_path):
    try:
        model = joblib.load(model_path)
        print(f"Trained model loaded successfully from '{model_path}'.")
        
        # Load the feature names the model was trained on
        if os.path.exists(feature_names_path):
            with open(feature_names_path, 'r') as f:
                trained_model_features = json.load(f)
            print(f"Loaded feature names model was trained on from '{feature_names_path}'. Features: {trained_model_features[:5]}... ({len(trained_model_features)} total)")
        elif hasattr(model, 'feature_names_in_'): # Fallback for some sklearn versions/models
            trained_model_features = model.feature_names_in_.tolist()
            print(f"Retrieved feature names from model attribute 'feature_names_in_'. Features: {trained_model_features[:5]}... ({len(trained_model_features)} total)")
        else:
            print(f"Warning: Feature names file '{feature_names_path}' not found, and model does not have 'feature_names_in_'. Feature order and naming consistency will be critical and is assumed.")
            
    except Exception as e:
        print(f"Error loading model from '{model_path}': {e}")
        model = None
else:
    print(f"Error: Model file not found at '{model_path}'. Make sure the training pipeline has been run.")

## 3. Load Feature Data for Inference

For this exercise, we are using the entire `bank-features-selected.csv` as a new batch of data for inference. In a real-world scenario, this would be new, unseen data that has undergone the same preprocessing steps as the training data (handled by the feature pipeline). The `bank-features-selected.csv` should ideally contain only the features the model was trained on and in the correct order.

In [ ]:
inference_data_path = '../feature_pipeline/bank-features-selected.csv'
X_inference = None
inference_df_for_output = None # To store data for joining with predictions

if os.path.exists(inference_data_path) and model is not None:
    try:
        inference_df = pd.read_csv(inference_data_path)
        print(f"Loaded inference data from '{inference_data_path}'. Shape: {inference_df.shape}")
        inference_df_for_output = inference_df.copy() # Keep a copy for output
        
        # If 'y' (target) column is present, remove it for X_inference
        if 'y' in inference_df.columns:
            X_inference = inference_df.drop(columns=['y'])
            print("Removed target column 'y' from inference data for X_inference.")
        else:
            X_inference = inference_df.copy()
            print("Target column 'y' not found in inference data, using all columns as features for X_inference.")
        
        # Ensure feature names and order match what the model was trained on
        if trained_model_features: # If we have the list of features the model was trained on
            missing_cols_in_data = set(trained_model_features) - set(X_inference.columns)
            extra_cols_in_data = set(X_inference.columns) - set(trained_model_features)
            
            if missing_cols_in_data:
                print(f"Error: The following features expected by the model are MISSING in the inference data: {missing_cols_in_data}")
                X_inference = None # Invalidate X_inference
            elif extra_cols_in_data:
                print(f"Warning: The following features in inference data were NOT expected by the model and will be REMOVED: {extra_cols_in_data}")
                X_inference = X_inference[trained_model_features] # Select only expected features in correct order
                print(f"Features for inference (X_inference) prepared. Shape: {X_inference.shape}")
            else: # Columns match exactly (or only need reordering)
                X_inference = X_inference[trained_model_features]
                print(f"Features for inference (X_inference) prepared. Shape: {X_inference.shape}")
                print("Columns confirmed/reordered to match model's training feature order.")
        elif X_inference is not None: 
            print(f"Warning: Feature names model was trained on are not definitively known. Assuming columns in '{inference_data_path}' are correct and in order.")
            print(f"Features for inference (X_inference) prepared. Shape: {X_inference.shape}")
            
    except Exception as e:
        print(f"Error loading or processing inference data: {e}")
        X_inference = None
else:
    if model is None:
        print("Model not loaded. Cannot proceed with loading inference data.")
    else:
        print(f"Error: Inference data file not found at '{inference_data_path}'.")

## 4. Generate Predictions

In [ ]:
predictions_labels = None
predictions_probabilities = None

if model is not None and X_inference is not None:
    print("Generating predictions...")
    try:
        predictions_labels = model.predict(X_inference)
        print("Generated class label predictions.")
        
        if hasattr(model, 'predict_proba'):
            predictions_probabilities = model.predict_proba(X_inference)[:, 1] # Probability of positive class (1)
            print("Generated prediction probabilities for the positive class.")
        else:
            print("Model does not support predict_proba(). Skipping probability predictions.")
            
    except ValueError as ve:
        print(f"ValueError during prediction: {ve}")
        print("This often occurs if features in inference data don't match model expectations (names, order, type, or count).")
        print(f"Model expected features (count: {len(trained_model_features)}): {trained_model_features[:10]}...")
        print(f"Actual features provided (count: {len(X_inference.columns)}): {X_inference.columns.tolist()[:10]}...")
        # X_inference.info()
    except Exception as e:
        print(f"General error during prediction: {e}")
else:
    print("Model or inference data not available/prepared. Skipping predictions.")

## 5. Save Predictions

In [ ]:
output_predictions_path = 'predictions.csv'

if predictions_labels is not None and inference_df_for_output is not None:
    print(f"Saving predictions to '{output_predictions_path}'...")
    try:
        # Create a DataFrame for predictions. 
        # We'll use 'inference_df_for_output' which is a copy of the originally loaded inference data.
        # This ensures we keep all original columns for context and add prediction columns.
        # It's important that 'predictions_labels' and 'predictions_probabilities' align with 'inference_df_for_output'.
        # This alignment is implicitly handled if X_inference was derived correctly from inference_df and no rows were dropped from X_inference itself post-load.
        # If X_inference had rows dropped that were not in inference_df_for_output, this would error or misalign.
        # Our current logic for X_inference creation (dropping 'y' and reordering/selecting columns based on trained_model_features)
        # should maintain row integrity relative to the original loaded inference_df.

        results_df = inference_df_for_output.copy() # Start with a copy of the original data loaded for inference
        results_df['predicted_label'] = predictions_labels
        
        if predictions_probabilities is not None:
            results_df['prediction_probability_yes'] = predictions_probabilities
            
        results_df.to_csv(output_predictions_path, index=False)
        print(f"Predictions saved successfully to '{output_predictions_path}'.")
        print("\nFirst 5 rows of the predictions file:")
        display(results_df.head())
        
    except Exception as e:
        print(f"Error saving predictions: {e}")
else:
    print("Predictions or original inference data for output not available. Skipping saving predictions.")

### Content of `predictions.csv`

The `predictions.csv` file contains the original data from the input batch (`bank-features-selected.csv`) along with the following new columns:

-   `predicted_label`: The predicted class label for each instance (0 for 'no', 1 for 'yes' - indicating whether the customer is predicted to subscribe to a term deposit).
-   `prediction_probability_yes` (optional): If the model supports probability predictions and they were generated, this column contains the probability of the instance belonging to the positive class ('yes'). This provides a measure of confidence in the prediction.